In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "tasks" / "tasks_100.json").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing src/tasks/tasks_100.json")


def _load_task_env(repo_root: Path, task_id: str) -> dict:
    tasks_path = repo_root / "src" / "tasks" / "tasks_100.json"
    tasks = json.loads(tasks_path.read_text(encoding="utf-8"))
    for task in tasks:
        if task.get("task_id") == task_id:
            world = task.get("world", {})
            return {
                "task_id": task.get("task_id"),
                "task_type": task.get("task_type"),
                "obstacles": world.get("obstacles", []),
                "targets": world.get("targets", []),
            }
    raise ValueError(f"Task not found in tasks_100.json: {task_id}")


def _parse_tree_json(raw: str) -> dict:
    text = (raw or "").strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return json.loads(text)


def _normalize_angle_deg(angle: float) -> float:
    return ((angle + 180.0) % 360.0) - 180.0


def _simulate_actions(start_pose: dict, actions: list[tuple[str, dict]]) -> tuple[list[tuple[float, float]], float]:
    x = float(start_pose.get("x", 0.0))
    y = float(start_pose.get("y", 0.0))
    heading_deg = float(start_pose.get("heading", 0.0))
    points = [(x, y)]

    for tool_name, arguments in actions:
        if tool_name == "rotate_spot":
            heading_deg += float(arguments.get("degrees", 0.0))
        elif tool_name == "move_spot":
            meters = float(arguments.get("meters", 0.0))
            heading_rad = math.radians(heading_deg)
            x += meters * math.cos(heading_rad)
            y += meters * math.sin(heading_rad)
            points.append((x, y))

    return points, heading_deg


def _beam_prune(outcomes: list[dict], max_items: int = 300) -> list[dict]:
    ranked = sorted(outcomes, key=lambda o: (o.get("penalty", 0.0), len(o.get("actions", []))))
    return ranked[:max_items]


def _merge_outcomes(base: dict, delta: dict) -> dict:
    return {
        "status": delta.get("status", False),
        "actions": (base.get("actions", []) + delta.get("actions", [])),
        "penalty": float(base.get("penalty", 0.0)) + float(delta.get("penalty", 0.0)),
    }


def _eval_node(node: dict, obs_hints: dict, max_outcomes: int = 300) -> list[dict]:
    node_type = node.get("type")

    if node_type == "action":
        call = node.get("call", {}) or {}
        tool_name = call.get("tool_name")
        args = call.get("arguments", {}) or {}
        if tool_name:
            return [{"status": True, "actions": [(tool_name, args)], "penalty": 0.0}]
        return [{"status": True, "actions": [], "penalty": 0.0}]

    if node_type == "condition":
        obs_name = node.get("observation")
        expected = bool(node.get("expected"))
        hinted = obs_hints.get(obs_name)

        if hinted is None:
            return [
                {"status": True, "actions": [], "penalty": 0.2},
                {"status": False, "actions": [], "penalty": 0.2},
            ]

        hinted = bool(hinted)
        if hinted == expected:
            return [
                {"status": True, "actions": [], "penalty": 0.0},
                {"status": False, "actions": [], "penalty": 2.0},
            ]

        return [
            {"status": True, "actions": [], "penalty": 2.0},
            {"status": False, "actions": [], "penalty": 0.0},
        ]

    children = node.get("children", []) or []

    if node_type == "sequence":
        active = [{"status": True, "actions": [], "penalty": 0.0}]

        for child in children:
            child_outcomes = _eval_node(child, obs_hints, max_outcomes=max_outcomes)
            next_active = []

            for base in active:
                if not base["status"]:
                    next_active.append(base)
                    continue

                for child_out in child_outcomes:
                    merged = _merge_outcomes(base, child_out)
                    if child_out["status"]:
                        merged["status"] = True
                        next_active.append(merged)
                    else:
                        merged["status"] = False
                        next_active.append(merged)

            active = _beam_prune(next_active, max_items=max_outcomes)

        return active

    if node_type == "fallback":
        carry = [{"status": False, "actions": [], "penalty": 0.0}]  # failed so far; keep trying next child
        successes = []

        for child in children:
            child_outcomes = _eval_node(child, obs_hints, max_outcomes=max_outcomes)
            next_carry = []

            for base in carry:
                for child_out in child_outcomes:
                    merged = _merge_outcomes(base, child_out)
                    if child_out["status"]:
                        merged["status"] = True
                        successes.append(merged)
                    else:
                        merged["status"] = False
                        next_carry.append(merged)

            carry = _beam_prune(next_carry, max_items=max_outcomes)
            successes = _beam_prune(successes, max_items=max_outcomes)

        return _beam_prune(successes + carry, max_items=max_outcomes)

    # Unknown node type: neutral success with no actions.
    return [{"status": True, "actions": [], "penalty": 0.0}]


def _infer_best_path_from_tree(
    tree_root: dict,
    start_pose: dict,
    recorded_final: dict,
    obs_hints: dict,
) -> tuple[list[tuple[float, float]], float, float]:
    outcomes = _eval_node(tree_root, obs_hints, max_outcomes=400)

    fx = float(recorded_final.get("x", start_pose.get("x", 0.0)))
    fy = float(recorded_final.get("y", start_pose.get("y", 0.0)))
    fheading = float(recorded_final.get("heading", start_pose.get("heading", 0.0)))

    best_score = float("inf")
    best_points = [(float(start_pose.get("x", 0.0)), float(start_pose.get("y", 0.0)))]
    best_end_heading = float(start_pose.get("heading", 0.0))

    for out in outcomes:
        points, end_heading = _simulate_actions(start_pose, out.get("actions", []))
        ex, ey = points[-1]

        dist = math.hypot(fx - ex, fy - ey)
        hdiff = abs(_normalize_angle_deg(end_heading - fheading))

        score = dist + 0.03 * hdiff + 0.25 * float(out.get("penalty", 0.0))

        if score < best_score:
            best_score = score
            best_points = points
            best_end_heading = end_heading

    return best_points, best_end_heading, best_score


def plot_task_world_and_tree_actions(folder: str, model: str, task_id: str) -> None:
    """Draw task world and branch-inferred per-tree trajectories."""
    repo_root = _find_repo_root(Path.cwd())
    task_env = _load_task_env(repo_root, task_id)

    run_dir = repo_root / folder
    task_json = run_dir / "tasks" / f"task_{task_id}.json"

    if not task_json.exists():
        raise FileNotFoundError(f"Missing task file: {task_json}")

    task_result = json.loads(task_json.read_text(encoding="utf-8"))

    fig, ax = plt.subplots(figsize=(8, 6))

    for obstacle in task_env["obstacles"]:
        rect = Rectangle(
            (obstacle["x1"], obstacle["y1"]),
            obstacle["x2"] - obstacle["x1"],
            obstacle["y2"] - obstacle["y1"],
            facecolor="black",
            edgecolor="black",
            alpha=0.30,
        )
        ax.add_patch(rect)

    for target in task_env["targets"]:
        rect = Rectangle(
            (target["x1"], target["y1"]),
            target["x2"] - target["x1"],
            target["y2"] - target["y1"],
            facecolor="limegreen",
            edgecolor="green",
            alpha=0.30,
        )
        ax.add_patch(rect)

    start_pose = {"x": 0.0, "y": 0.0, "heading": 0.0}
    ax.scatter([0.0], [0.0], c="blue", marker="o", s=70, label="start")

    trees = task_result.get("behavior_trees", [])
    colors = plt.cm.tab10.colors

    for idx, bt in enumerate(trees):
        color = colors[idx % len(colors)]
        raw_tree = bt.get("decoder_output", "")

        recorded_final = (bt.get("plan_results") or {}).get("final_spot") or start_pose
        obs_hints = (bt.get("plan_results") or {}).get("observations") or {}

        try:
            tree_obj = _parse_tree_json(raw_tree)
            points, inferred_heading, _ = _infer_best_path_from_tree(
                tree_obj.get("root", {}),
                start_pose,
                recorded_final,
                obs_hints,
            )
        except Exception:
            points = [(float(start_pose["x"]), float(start_pose["y"]))]
            inferred_heading = float(start_pose["heading"])

        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        ax.plot(xs, ys, color=color, linewidth=2.2, alpha=0.9, label=f"tree {idx + 1}")

        fx = float(recorded_final.get("x", points[-1][0]))
        fy = float(recorded_final.get("y", points[-1][1]))
        ax.scatter([fx], [fy], c=[color], marker="x", s=70)

        # If inferred path endpoint is still off, show residual mismatch lightly.
        sx, sy = points[-1]
        if abs(fx - sx) > 1e-6 or abs(fy - sy) > 1e-6:
            ax.plot([sx, fx], [sy, fy], linestyle="--", color=color, alpha=0.35)

        start_pose = {
            "x": fx,
            "y": fy,
            "heading": float(recorded_final.get("heading", inferred_heading)),
        }

    final_spot = task_result.get("final_spot", start_pose)
    ax.scatter([final_spot.get("x", 0.0)], [final_spot.get("y", 0.0)], c="red", marker="*", s=120, label="task final")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.set_title(f"{task_id} | {model}")
    ax.legend(loc="best", fontsize=9)
    plt.show()


In [ ]:
plot_task_world_and_tree_actions(
    folder="results/pvd_pairs_t0.3/granite-3.3-8b-instruct__Qwen2.5-3B-Instruct",
    model="granite-3.3-8b-instruct__Qwen2.5-3B-Instruct",
    task_id="go_to_target_v20",
)


In [ ]:
plot_task_world_and_tree_actions(
    folder="results/pvd_pairs_t0.6/granite-3.3-8b-instruct__Qwen2.5-1.5B-Instruct",
    model="granite-3.3-8b-instruct__Qwen2.5-1.5B-Instruct",
    task_id="face_target_v20",
)


In [ ]:
plot_task_world_and_tree_actions(
    folder="results/pvd_bt_all100_temp0.5/phi-4",
    model="phi-4",
    task_id="move_to_closest_target_v20",
)


In [ ]:
plot_task_world_and_tree_actions(
    folder="results/pvd_bt_all100_temp0.3/granite-3.3-8b-instruct",
    model="granite-3.3-8b-instruct",
    task_id="go_to_multiple_targets_v13",
)


In [ ]:
plot_task_world_and_tree_actions(
    folder="results/pvd_pairs_t0.0/Qwen3-8B__Qwen2.5-3B-Instruct",
    model="Qwen3-8B__Qwen2.5-3B-Instruct",
    task_id="go_around_obstacle_v14",
)
